In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [ ]:
df=pd.read_csv("results_gpt_math.csv")
print(f"Total rows in CSV: {len(df)}")

Total rows in CSV: 50


In [ ]:
def fetch_html_body_content(html_url):

    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("⚠ HTML 请求失败:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # 移除 Abstract
    text = re.sub(
        r'Abstract[:\s].*?(?=(Introduction|1\.\sIntroduction))',
        '',
        text,
        flags=re.S|re.I
    )

    # 截断 Acknowledgment
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]

    for pat in ack_patterns:
        match = re.search(pat, text, re.I)
        if match:
            text = text[:match.start()]
            break

    text = "\n".join(line for line in text.splitlines() if line.strip())

    return text, "html_success"

In [ ]:
df["gpt_rate"] = pd.NA
df["gpt_explanation"] = ""

# --- DATA QUALITY FIX: strip arXiv HTML navigation boilerplate ---
# arXiv HTML pages start with a big block of UI elements and a table-of-contents
# (section number + title lines) before the actual article prose. We need to skip
# past all of that to give Llama clean article text.

# Exact strings that appear in the arXiv HTML nav/UI preamble
BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "×",
}

# Substrings — if ANY of these appear anywhere in the line, skip it
BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue",
    "Back to arXiv",
    "Why HTML?",
    "Report Issue",
    "Submit without GitHub",
    "Submit in GitHub",
    "Back to Introduction",
    "Back to ",            # catches "Back to <any section>"
    "Content selection saved",
    "Describe the issue below",
]

def get_article_snippet(full_text, max_chars=4000):
    """
    Strip the arXiv HTML preamble (nav UI + table of contents), then return
    the first max_chars of actual article prose.

    Strategy: scan forward until we find a line that looks like the start of
    real article content (a sentence-length line with lowercase words — i.e.
    actual prose, not a short section heading). Everything before that is
    preamble/TOC and gets dropped.
    """
    lines = full_text.splitlines()

    # Phase 1: Find where the real article prose begins.
    # TOC lines are short (section titles). Real prose lines are longer and
    # contain lowercase words forming sentences.
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        # Skip blanks
        if not stripped:
            continue
        # A "prose" line: reasonably long, contains spaces (multiple words),
        # and has some lowercase characters (not all-caps headings).
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2: From body_start onward, drop any remaining boilerplate lines
    cleaned_lines = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            # Preserve paragraph breaks in the body, but skip leading blanks
            if cleaned_lines:
                cleaned_lines.append("")
            continue
        # Skip lines matching known boilerplate substrings
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        # Skip exact boilerplate matches
        if stripped in BOILERPLATE_EXACT:
            continue
        # Skip bare section numbers like "2", "2.1", "A.3"
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    return cleaned[:max_chars]

In [ ]:
# gpt client
gpt_client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# 循环处理每一行
for index, row in df.iterrows():
    try:
        # Unpack the tuple: text goes to 'raw_text', status goes to '_'
        raw_text, status = fetch_html_body_content(row["html_url"])

        if raw_text is None:
            print(f"Skipping row {index} due to fetch error: {status}")
            continue

        article = get_article_snippet(raw_text)
        summary = row["gpt_prompt"]
        prompt = f"""
Evaluate the quality of summaries written for a news article.
Rate each summary on four dimensions:
style transfer intensity, content preservation, naturalness and readability.
You should rate on a scale from 1 (worst) to 5 (best).

Article: {article}
Summary: {summary}

Provide your evaluation as follows:
Explanation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Explanation:' and 'Total rating:' in your answer.
"""

        response = gpt_client.chat.completions.create(
            model="gpt-5.2",
            messages=[{"role": "user", "content": prompt}],
            stream=False
        )
        output_text = response.choices[0].message.content
        print(index, ":")
        print(output_text)

        # 从 GPT 输出中提取 Total rating 和 Explanation
        lines = output_text.splitlines()
        explanation = ""
        rating = None
        for line in lines:
            if line.lower().startswith("explanation:"):
                explanation = line[len("Explanation:"):].strip()
            elif line.lower().startswith("total rating:"):
                rating_str = line[len("Total rating:"):].strip()
                try:
                    rating = int(rating_str)
                except:
                    rating = None

        df.at[index, "gpt_rate"] = rating
        df.at[index, "gpt_explanation"] = explanation

    except Exception as e:
        print(f"Error at row {index}: {e}")

# 保存回 CSV
df.to_csv("results_gpt_math_gpt_ratings.csv", index=False)

0 :
Explanation: The “summary” is essentially a meta-commentary explaining that the input article contains no substantive content (only section headings like “Preliminaries and proofs” and “Proof of Theorem”). Given the article itself is empty of meaningful information, the summary preserves what is actually present and accurately notes the lack of research details. However, it does not perform a true news-style summarization, and there is no style transfer beyond a clear, explanatory/helpdesk tone. It is well-written, coherent, and readable, but only partially fulfills the intent of summarizing an article.

Total rating: 4
1 :
Explanation: The summary is clear, well-structured, and reads like a natural academic-popular explanation. It preserves many of the visible section-level topics from the article outline (notation, case splits, bounding number of colors, vertices with exactly three incident colors, finding a rainbow \(K_4\), characterization of small configurations, extensions). 

In [ ]:
# 统计每个评分数量
df=pd.read_csv("results_gpt_math_gpt_ratings.csv")
rating_counts = df["gpt_rate"].value_counts().sort_index()
print("\nRating counts (1–5):")
for i in range(1, 6):
    count = rating_counts.get(i, 0)
    print(f"{i}: {count}")


Rating counts (1–5):
1: 0
2: 1
3: 1
4: 18
5: 29
